This is a "stronger" MLP implementation for solving the MNIST multi-class classification task. This this, you could stably reach something around **98% of accuracy**. More could be done (even still on MLPs and not CNNs), but we stop here as this is students-readable and educationally useful.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Full training images:", x_train_full.shape)
print("Full training labels:", y_train_full.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

In [ ]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Minimum pixel value:", x_train_full.min())
print("Maximum pixel value:", x_train_full.max())

In [ ]:
validation_size = 10_000

x_val = x_train_full[-validation_size:]
y_val = y_train_full[-validation_size:]

x_train = x_train_full[:-validation_size]
y_train = y_train_full[:-validation_size]

print("Training images:", x_train.shape)
print("Validation images:", x_val.shape)
print("Test images:", x_test.shape)

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def build_strong_mlp(input_shape=(28, 28), num_classes=10):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Flatten(),

        layers.Dense(512, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),

        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),

        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),

        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_strong_mlp()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
print(len(history.history["loss"]))
print(len(history.history["accuracy"]))

In [ ]:
def plot_history(history):
    history_dict = history.history

    plt.figure(figsize=(7, 5))
    plt.plot(history_dict["loss"], label="Training loss")
    plt.plot(history_dict["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and validation loss")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(history_dict["accuracy"], label="Training accuracy")
    plt.plot(history_dict["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and validation accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_history(history)

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("First 20 true labels:     ", y_test[:20])
print("First 20 predicted labels:", y_pred[:20])

In [ ]:
def show_predictions(x, y_true, y_pred, correct=True, n=10):
    if correct:
        indices = np.where(y_true == y_pred)[0]
        title_prefix = "Correct"
    else:
        indices = np.where(y_true != y_pred)[0]
        title_prefix = "Incorrect"

    selected = indices[:n]

    plt.figure(figsize=(10, 3))
    for i, idx in enumerate(selected):
        plt.subplot(2, 5, i + 1)
        plt.imshow(x[idx], cmap="gray")
        plt.title(f"True: {y_true[idx]} | Pred: {y_pred[idx]}")
        plt.axis("off")
    plt.suptitle(f"{title_prefix} predictions")
    plt.tight_layout()
    plt.show()

show_predictions(x_test, y_test, y_pred, correct=True, n=10)
show_predictions(x_test, y_test, y_pred, correct=False, n=10)

In [ ]:
try:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(ax=ax, values_format="d")
    plt.title("Confusion matrix")
    plt.show()
except ImportError:
    print("scikit-learn is not installed. Install it to display the confusion matrix.")

In [ ]:
model.save("strong_mnist_mlp.keras")
print("Model saved as strong_mnist_mlp.keras")